# Hyperparameter Search — Validation Set Only

Selects the XGBoost and LightGBM configurations used by `03_xgboost_sybil`, `04_lightgbm_sybil`, and the
robustness notebooks (`07`, `08`, `09`). The model settings that are not tuned come from `sp.xgb_model` and
`sp.lgbm_model`, the same definitions the model notebooks use.

**Protocol**
- Split: the stratified **group** split (`sp.make_splits(..., method='group')`), the revision's
  primary evaluation. The same selected configuration is reused in `11_split_comparison`.
- The test partition is removed before any model is fit; this notebook never sees test labels.
- Each configuration is trained once (seed 42) on the upsampled training partition, with early
  stopping on validation log-loss (patience 50, ceiling 5,000 rounds).
- **Selection criterion**: validation F1 at the F1-maximising validation threshold. Ties are broken
  by validation average precision (AP).

**Search space** (full grids over the ranges stated in the paper)

| Model | Stage | Grid | Configs |
|---|---|---|---|
| XGBoost | 1 | `learning_rate` {0.3, 0.1, 0.05} × `max_depth` {6, 8, 10, 12} × `subsample` {1.0, 0.7, 0.5} × `colsample_bytree` {1.0, 0.8} | 72 |
| XGBoost | 2 | `min_child_weight` {1, 5, 10} at the stage-1 best | +2 |
| LightGBM | — | `num_leaves` {31, 63, 127, 255, 511, 1023} × `learning_rate` {0.1, 0.05} × `subsample` {1.0, 0.7} (`subsample_freq=1`) × `reg_lambda` {0, 1}; `colsample_bytree` 0.8 | 48 |

Results are written after every configuration to `results/02_search_<model>.csv`, so an
interrupted run resumes where it stopped. The selected configurations go to
`results/02_hyperparameter_search.json`, which the model notebooks read.

In [1]:
import os, time, json, itertools, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, average_precision_score, log_loss
import lightgbm as lgb_lib
import sybil_pipeline as sp
warnings.filterwarnings('ignore')

DATA_DIR = './data'
RESULTS_DIR = './results'
SEED = sp.SEED
os.makedirs(RESULTS_DIR, exist_ok=True)

df, tfm, labeled_anchors = sp.build_master_df(DATA_DIR)
S = sp.make_splits(df, sp.FEATS, method='group', seed=SEED)   # selection is done on the group split
X_train, y_train = S['X_train'], S['y_train']
X_val,   y_val   = S['X_val'],   S['y_val']
for k in ['X_test', 'y_test', 'idx_test']:
    del S[k]                    # test partition is not available in this notebook
print(sp.split_summary(df, {**S, 'idx_test': []}).iloc[:2].to_string(index=False))

[1] L0 features: 434,786 addresses, 55 cols


[2] Gas provision: 604,365 edges before 2024-05-02 00:00:00 UTC (296 later edges dropped)


[3] Labeled addresses in provision network: 4,308


[4] Provider and tree features: 88 cols


[5] CEX/DEX merged: 90 cols


[6] Labels: Sybil=18,211 (4.19%)


[7] Chain features, taxonomy, funding groups (371,764 groups)


All 62 features present ✓  (38.5s)


     Split  Total  Sybil  NonSybil  SybilPct
     Train 213046   8924    204122      4.19
Validation  91305   3824     87481      4.19


In [2]:
def val_scores(probs):
    thr = sp.best_f1_threshold(y_val, probs)
    return dict(val_f1=f1_score(y_val, (probs >= thr).astype(int)),
                val_ap=average_precision_score(y_val, probs),
                val_logloss=log_loss(y_val, probs), val_threshold=thr)

def run_grid(name, configs, fit):
    """Fit every config not already in results/02_search_<name>.csv; return the full table."""
    path = os.path.join(RESULTS_DIR, f'02_search_{name}.csv')
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    keys = list(configs[0])
    seen = set(map(tuple, done[keys].astype(str).values)) if len(done) else set()
    for i, cfg in enumerate(configs, 1):
        if tuple(str(cfg[k]) for k in keys) in seen:
            continue
        t0 = time.time()
        probs, rounds = fit(cfg)
        row = {**cfg, **val_scores(probs), 'rounds': rounds, 'seconds': round(time.time() - t0, 1)}
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(path, index=False)
        print(f"[{i}/{len(configs)}] {cfg}  val_F1={row['val_f1']:.4f}  AP={row['val_ap']:.4f}  "
              f"rounds={rounds}  {row['seconds']}s", flush=True)
    for k in keys:                          # restore dtypes after CSV round-trip
        done[k] = done[k].astype(type(configs[0][k]))
    return done

def best(table):
    return table.sort_values(['val_f1', 'val_ap'], ascending=False).iloc[0]

INT_PARAMS = {'max_depth', 'min_child_weight', 'num_leaves'}

def as_params(row, keys):
    # results-table row -> hyperparameter dict with the right Python types
    return {k: int(row[k]) if k in INT_PARAMS else float(row[k]) for k in keys}

## XGBoost — stage 1

In [3]:
def fit_xgb(cfg):
    m = sp.xgb_model(cfg, SEED)   # fixed settings as in the model notebooks
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return m.predict_proba(X_val)[:, 1], int(m.best_iteration)

xgb_grid = [dict(learning_rate=lr, max_depth=d, subsample=s, colsample_bytree=c, min_child_weight=1)
            for lr, d, s, c in itertools.product([0.3, 0.1, 0.05], [6, 8, 10, 12],
                                                 [1.0, 0.7, 0.5], [1.0, 0.8])]
xgb1 = run_grid('xgboost', xgb_grid, fit_xgb)
print(xgb1.sort_values('val_f1', ascending=False).head(10).round(4).to_string(index=False))

[1/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6907  AP=0.7422  rounds=671  32.1s


[2/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.6913  AP=0.7385  rounds=746  20.7s


[3/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6897  AP=0.7348  rounds=628  21.0s


[4/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.6886  AP=0.7374  rounds=602  30.1s


[5/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 0.5, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6822  AP=0.7277  rounds=558  20.4s


...

[69/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6993  AP=0.7528  rounds=533  36.8s


[70/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.6977  AP=0.7563  rounds=568  40.4s


[71/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6990  AP=0.7537  rounds=526  35.4s


[72/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.7019  AP=0.7543  rounds=547  36.1s


 learning_rate  max_depth  subsample  colsample_bytree  min_child_weight  val_f1  val_ap  val_logloss  val_threshold  rounds  seconds
          0.05          8        0.7               1.0                 1  0.7032  0.7552       0.0826         0.5262    1679    118.3
          0.05         10        0.5               1.0                 1  0.7025  0.7527       0.0820         0.5450     906     50.3
          0.05         12        0.5               0.8                 1  0.7019  0.7543       0.0805         0.5808     547     36.1
          0.10         12        0.7               0.8                 1  0.7012  0.7525       0.0824         0.5899     297     21.3
          0.10         10        0.7               0.8                 1  0.7012  0.7531       0.0835         0.5398     481     26.6
          0.10          8        0.7               1.0                 1  0.7010  0.7514       0.0853         0.5760     855     37.7
          0.05          8        0.7               0.8        

## XGBoost — stage 2 (`min_child_weight`)

In [4]:
b1 = best(xgb1)
xgb_grid2 = xgb_grid + [{**as_params(b1, ['learning_rate', 'max_depth', 'subsample', 'colsample_bytree']),
                         'min_child_weight': w} for w in [5, 10]]
xgb_all = run_grid('xgboost', xgb_grid2, fit_xgb)
xgb_best = best(xgb_all)
XGB_SELECTED = as_params(xgb_best, ['learning_rate', 'max_depth', 'subsample',
                                    'colsample_bytree', 'min_child_weight'])
print('Selected XGBoost config:', XGB_SELECTED)
print(xgb_best[['val_f1', 'val_ap', 'rounds']].to_string())

[73/74] {'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 5}  val_F1=0.7010  AP=0.7541  rounds=1927  70.0s


[74/74] {'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 10}  val_F1=0.7016  AP=0.7518  rounds=2536  95.1s


Selected XGBoost config: {'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 1}
val_f1       0.703157
val_ap       0.755201
rounds    1679.000000


## LightGBM

In [5]:
def fit_lgbm(cfg):
    m = sp.lgbm_model(cfg, SEED)   # fixed settings as in the model notebooks
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb_lib.early_stopping(50, verbose=False), lgb_lib.log_evaluation(-1)])
    return m.predict_proba(X_val)[:, 1], int(m.best_iteration_)

lgbm_grid = [dict(num_leaves=nl, learning_rate=lr, subsample=s, reg_lambda=l2)
             for nl, lr, s, l2 in itertools.product([31, 63, 127, 255, 511, 1023], [0.1, 0.05],
                                                    [1.0, 0.7], [0.0, 1.0])]
lgbm_all = run_grid('lightgbm', lgbm_grid, fit_lgbm)
lgbm_best = best(lgbm_all)
LGBM_SELECTED = as_params(lgbm_best, ['num_leaves', 'learning_rate', 'subsample', 'reg_lambda'])
print('Selected LightGBM config:', LGBM_SELECTED)
print(lgbm_all.sort_values('val_f1', ascending=False).head(10).round(4).to_string(index=False))

[1/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 1.0, 'reg_lambda': 0.0}  val_F1=0.6871  AP=0.7376  rounds=2059  39.5s


[2/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 1.0, 'reg_lambda': 1.0}  val_F1=0.6948  AP=0.7481  rounds=2417  46.2s


[3/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 0.7, 'reg_lambda': 0.0}  val_F1=0.6991  AP=0.7436  rounds=1950  39.5s


[4/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 0.7, 'reg_lambda': 1.0}  val_F1=0.7008  AP=0.7506  rounds=2596  52.2s


[5/48] {'num_leaves': 31, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 0.0}  val_F1=0.6914  AP=0.7398  rounds=3899  74.7s


...

[45/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 0.0}  val_F1=0.6978  AP=0.7518  rounds=138  24.3s


[46/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0}  val_F1=0.7023  AP=0.7581  rounds=178  29.3s


[47/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 0.0}  val_F1=0.7006  AP=0.7547  rounds=142  25.2s


[48/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0}  val_F1=0.7039  AP=0.7590  rounds=164  27.7s


Selected LightGBM config: {'num_leaves': 127, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0}
 num_leaves  learning_rate  subsample  reg_lambda  val_f1  val_ap  val_logloss  val_threshold  rounds  seconds
        127           0.05        0.7         1.0  0.7067  0.7600       0.0788         0.4355    1277     48.8
        511           0.05        0.7         0.0  0.7061  0.7543       0.0787         0.5598     270     26.1
        255           0.05        0.7         1.0  0.7047  0.7587       0.0780         0.5464     592     35.2
       1023           0.05        0.7         1.0  0.7039  0.7590       0.0772         0.5715     164     27.7
         31           0.05        0.7         1.0  0.7038  0.7558       0.0816         0.5931    4749     97.0
         31           0.05        0.7         0.0  0.7036  0.7516       0.0833         0.5994    4321     83.4
        511           0.10        0.7         1.0  0.7033  0.7535       0.0798         0.5543     156     16.7
      

## Marginal effect of each hyperparameter (validation F1)

For each hyperparameter value: the best validation F1 over all configs with that value. This is
the ablation view of the grid.

In [6]:
def marginal(table, params):
    out = []
    for p in params:
        g = table.groupby(p)['val_f1'].max()
        out += [dict(param=p, value=v, best_val_f1=round(f, 4)) for v, f in g.items()]
    return pd.DataFrame(out)

print('XGBoost (stage 1 grid):')
print(marginal(xgb1, ['learning_rate', 'max_depth', 'subsample', 'colsample_bytree']).to_string(index=False))
print('\nLightGBM:')
print(marginal(lgbm_all, ['num_leaves', 'learning_rate', 'subsample', 'reg_lambda']).to_string(index=False))

XGBoost (stage 1 grid):
           param  value  best_val_f1
   learning_rate   0.05       0.7032
   learning_rate   0.10       0.7012
   learning_rate   0.30       0.6955
       max_depth   6.00       0.6970
       max_depth   8.00       0.7032
       max_depth  10.00       0.7025
       max_depth  12.00       0.7019
       subsample   0.50       0.7025
       subsample   0.70       0.7032
       subsample   1.00       0.6980
colsample_bytree   0.80       0.7019
colsample_bytree   1.00       0.7032

LightGBM:
        param   value  best_val_f1
   num_leaves   31.00       0.7038
   num_leaves   63.00       0.7025
   num_leaves  127.00       0.7067
   num_leaves  255.00       0.7047
   num_leaves  511.00       0.7061
   num_leaves 1023.00       0.7039
learning_rate    0.05       0.7067
learning_rate    0.10       0.7033
    subsample    0.70       0.7067
    subsample    1.00       0.7023
   reg_lambda    0.00       0.7061
   reg_lambda    1.00       0.7067


## Save selected configurations

In [7]:
sp.save_results([], '02_hyperparameter_search', extra=dict(
    selection='max validation F1 at validation-optimal threshold; ties by validation AP; seed 42',
    xgb_selected=XGB_SELECTED, lgbm_selected=LGBM_SELECTED,
    xgb_selected_val=dict(val_f1=float(xgb_best.val_f1), val_ap=float(xgb_best.val_ap), rounds=int(xgb_best.rounds)),
    lgbm_selected_val=dict(val_f1=float(lgbm_best.val_f1), val_ap=float(lgbm_best.val_ap), rounds=int(lgbm_best.rounds)),
    n_configs=dict(xgboost=len(xgb_all), lightgbm=len(lgbm_all)),
))

Saved results/05_hyperparameter_search_group.json
